# Phase 3: Workforce Intelligence — Steps 3.5 & 3.6: Course Recommendation Engine

This notebook implements the course recommendation system for employee skill gap remediation. We develop two approaches:
1. **Recommendation Engine v1**: A rule-based system using strict exact-match lookup from missing skills to courses.
2. **Recommendation Engine v2**: A semantic search system using `sentence-transformers` embeddings to map missing skills to the most contextually relevant courses.

In [1]:
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

proc_dir = os.path.join("data", "processed")
print(f"Processed directory: {os.path.abspath(proc_dir)}")

## 1. Course Catalog Synthesis
We define a catalog of 35 courses covering major O*NET-SOC skills. 

*(Note: This dataset is entirely synthetic and serves to link training content to O*NET target skills.)*

In [2]:
courses_data = [
    ("Foundations of Modern Science", "Science", "Intermediate", 14),
    ("Critical Thinking and Analytical Problem Solving", "Critical Thinking", "Intermediate", 7),
    ("Effective Communication and Active Listening", "Active Listening", "Beginner", 5),
    ("Essential Mathematics for Business", "Mathematics", "Beginner", 10),
    ("Effective Learning and Study Strategies", "Learning Strategies", "Beginner", 4),
    ("Performance Monitoring and Quality Assurance", "Monitoring", "Intermediate", 6),
    ("Advanced Microsoft Excel Masterclass", "Microsoft Excel", "Advanced", 12),
    ("Public Speaking and Presentation Skills", "Speaking", "Intermediate", 5),
    ("Professional Document Creation in Microsoft Word", "Microsoft Word", "Beginner", 3),
    ("Business Writing Essentials", "Writing", "Beginner", 4),
    ("Speed Reading and Comprehension", "Reading Comprehension", "Beginner", 4),
    ("Continuous Learning Strategies", "Active Learning", "Beginner", 3),
    ("Designing High-Impact PowerPoint Presentations", "Microsoft PowerPoint", "Intermediate", 5),
    ("Microsoft Office Suite Productivity Boost", "Microsoft Office software", "Beginner", 8),
    ("Relational Database Foundations", "Database software", "Beginner", 7),
    ("IBM Notes Collaboration and Email", "IBM Notes", "Intermediate", 5),
    ("FileMaker Pro Database Development", "FileMaker Pro", "Intermediate", 10),
    ("Email Management with Microsoft Outlook", "Microsoft Outlook", "Beginner", 3),
    ("Data Analysis and Quality Statistics with Minitab", "Minitab", "Advanced", 10),
    ("Project Planning and Scheduling with MS Project", "Microsoft Project", "Advanced", 14),
    ("Negotiation and Influence Strategies", "Negotiation", "Intermediate", 6),
    ("Time Management and Personal Productivity", "Time Management", "Beginner", 2),
    ("Customer Service Excellence", "Service Orientation", "Beginner", 4),
    ("Troubleshooting and Technical Support", "Troubleshooting", "Intermediate", 8),
    ("SQL Querying for Beginners", "Database user interface and query software", "Beginner", 6),
    ("JavaScript Web Development Essentials", "JavaScript", "Intermediate", 12),
    ("Python Programming for Data Analytics", "Python", "Intermediate", 14),
    ("Linux System Administration Basics", "Linux", "Intermediate", 10),
    ("C++ Object-Oriented Programming", "C++", "Advanced", 20),
    ("HTML5 and CSS3 Responsive Web Design", "HTML", "Beginner", 5),
    ("Enterprise Resource Planning with SAP", "SAP", "Advanced", 15),
    ("Statistical Quality Control and Analysis", "Quality Control Analysis", "Advanced", 12),
    ("Complex Decision Making and Judgment", "Judgment and Decision Making", "Advanced", 8),
    ("Operations Analysis and Process Improvement", "Operations Analysis", "Advanced", 10),
    ("Systems Analysis and Design Foundations", "Systems Analysis", "Intermediate", 8)
]

df_courses = pd.DataFrame(courses_data, columns=["course_title", "target_skill", "difficulty", "duration_days"])
courses_path = os.path.join(proc_dir, "courses.csv")
df_courses.to_csv(courses_path, index=False)
print(f"Saved courses.csv successfully to {courses_path}")

Saved courses.csv successfully!


## 2. Load Employee Skill Gaps

In [3]:
df_gaps = pd.read_csv(os.path.join(proc_dir, "employee_skill_gaps.csv"))
print(f"Employee Gaps Shape: {df_gaps.shape}")

Employee Gaps Shape: (48468, 5)


## 3. Recommendation Engine v1 (if/else exact mapping)
We recommend up to 3 courses per employee based on an exact-match lookup of their missing skills (sorted by importance score descending).

In [4]:
skill_to_course = dict(zip(df_courses["target_skill"], df_courses["course_title"]))
df_gaps_sorted = df_gaps.sort_values(by=["employee_id", "importance_score"], ascending=[True, False])

recommendations_v1 = []
grouped_gaps = df_gaps_sorted.groupby("employee_id")

for emp_id, group in grouped_gaps:
    rec_courses = []
    for idx, row in group.iterrows():
        skill = row["missing_skill"]
        if skill in skill_to_course:
            course = skill_to_course[skill]
            if course not in rec_courses:
                rec_courses.append(course)
            if len(rec_courses) >= 3:
                break
                
    for course in rec_courses:
        recommendations_v1.append({
            "employee_id": emp_id,
            "recommended_course": course,
            "engine_version": "v1"
        })

df_rec_v1 = pd.DataFrame(recommendations_v1)
print(f"Generated {df_rec_v1.shape[0]} recommendations.")
print(df_rec_v1.head(10))

Generated 4409 recommendations.
   employee_id                                recommended_course engine_version
0            1           Email Management with Microsoft Outlook             v1
1            1   Project Planning and Scheduling with MS Project             v1
2            1  Critical Thinking and Analytical Problem Solving             v1
3            2             Python Programming for Data Analytics             v1
4            2    Designing High-Impact PowerPoint Presentations             v1
5            2  Critical Thinking and Analytical Problem Solving             v1
6            4         Microsoft Office Suite Productivity Boost             v1
7            4  Professional Document Creation in Microsoft Word             v1
8            4           Email Management with Microsoft Outlook             v1
9            5   Project Planning and Scheduling with MS Project             v1


## 4. Recommendation Engine v2 (Semantic Matching)
We encode all course entries (target_skill + course_title) and all unique missing skills using a pre-trained SentenceTransformer model, compute cosine similarity, and recommend the top 3 courses per employee.

In [5]:
print("Loading SentenceTransformer model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

# Embed courses
course_texts = [f"{row['target_skill']} - {row['course_title']}" for idx, row in df_courses.iterrows()]
course_embeddings = model.encode(course_texts, convert_to_tensor=True)

# Embed unique missing skills to save computation
unique_missing_skills = df_gaps["missing_skill"].unique().tolist()
skill_embeddings = model.encode(unique_missing_skills, convert_to_tensor=True)

# Compute similarities
similarities = cos_sim(skill_embeddings, course_embeddings).cpu().numpy()
skill_to_sims = {skill: similarities[i] for i, skill in enumerate(unique_missing_skills)}

recommendations_v2 = []
for emp_id, group in df_gaps.groupby("employee_id"):
    emp_skills = group["missing_skill"].tolist()
    if not emp_skills:
        continue
    
    # Get max similarity for each course across all of this employee's missing skills
    emp_sims = np.array([skill_to_sims[skill] for skill in emp_skills])
    max_course_sims = np.max(emp_sims, axis=0)
    
    # Rank top 3
    top_indices = np.argsort(max_course_sims)[::-1][:3]
    
    for rank_idx, idx in enumerate(top_indices):
        recommendations_v2.append({
            "employee_id": emp_id,
            "recommended_course": df_courses.iloc[idx]["course_title"],
            "similarity_score": float(max_course_sims[idx]),
            "rank": rank_idx + 1,
            "engine_version": "v2"
        })

df_rec_v2 = pd.DataFrame(recommendations_v2)
print(f"Generated {df_rec_v2.shape[0]} semantic recommendations.")
print(df_rec_v2.head(15))

Generated 4410 semantic recommendations.
    employee_id  ... engine_version
0             1  ...             v2
1             1  ...             v2
2             1  ...             v2
3             2  ...             v2
4             2  ...             v2
5             2  ...             v2
6             4  ...             v2
7             4  ...             v2
8             4  ...             v2
9             5  ...             v2
10            5  ...             v2
11            5  ...             v2
12            7  ...             v2
13            7  ...             v2
14            7  ...             v2

[15 rows x 5 columns]


## 5. Pivot & Save Recommendations for Lookup
We save both long format and pivoted format (one row per employee with columns: `recommended_course_1`, `recommended_course_2`, `recommended_course_3`) for easy downstream join in Step 3.7.

In [6]:
# Save raw recommendations
rec_v2_path = os.path.join(proc_dir, "employee_course_recommendations_v2.csv")
df_rec_v2.to_csv(rec_v2_path, index=False)

# Pivot recommendations
df_pivoted = df_rec_v2.pivot(index="employee_id", columns="rank", values="recommended_course").reset_index()
df_pivoted.columns = ["employee_id", "recommended_course_1", "recommended_course_2", "recommended_course_3"]

pivoted_path = os.path.join(proc_dir, "employee_recommendations_pivoted.csv")
df_pivoted.to_csv(pivoted_path, index=False)

print(f"Saved pivoted recommendations successfully to {pivoted_path}:")
print(df_pivoted.head(10))

Saved pivoted recommendations successfully!
   employee_id  ...                          recommended_course_3
0            1  ...       Email Management with Microsoft Outlook
1            2  ...            Essential Mathematics for Business
2            4  ...             IBM Notes Collaboration and Email
3            5  ...  Performance Monitoring and Quality Assurance
4            7  ...       Effective Learning and Study Strategies
5            8  ...                Continuous Learning Strategies
6           10  ...      Statistical Quality Control and Analysis
7           11  ...                Continuous Learning Strategies
8           12  ...       Effective Learning and Study Strategies
9           13  ...       Email Management with Microsoft Outlook

[10 rows x 4 columns]
